In [27]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re
import pyproj
import folium

In [31]:
# Override manual
semana_actual = 202617

In [32]:
# Opcional (similar a scipen = 999 en R, evita notación científica)
pd.set_option('display.float_format', '{:.0f}'.format)

# Importar calendario de festivos ------------------------------------------

festivos = pd.to_datetime([
    "2026-01-01", "2026-01-12",
    "2026-03-23", "2026-04-02", "2026-04-03",
    "2026-05-01", "2026-05-18", "2026-06-08",
    "2026-06-15", "2026-06-29", "2026-07-20",
    "2026-08-07", "2026-08-17", "2026-10-12",
    "2026-11-02", "2026-11-16", "2026-12-08",
    "2026-12-25"
])

print(festivos)

DatetimeIndex(['2026-01-01', '2026-01-12', '2026-03-23', '2026-04-02',
               '2026-04-03', '2026-05-01', '2026-05-18', '2026-06-08',
               '2026-06-15', '2026-06-29', '2026-07-20', '2026-08-07',
               '2026-08-17', '2026-10-12', '2026-11-02', '2026-11-16',
               '2026-12-08', '2026-12-25'],
              dtype='datetime64[ns]', freq=None)


In [34]:
calendario = pd.read_excel('C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/calendario_2026.xlsx')

calendario.head()

,DateKey,Date,DayOfMonth,DaySuffix,DayName,DayOfWeek,DayOfWeekInMonth,DayOfWeekInYear,DayOfQuarter,DayOfYear,...,LastDayOfQuarter,FirstDayOfYear,LastDayOfYear,FirstDayOfNextMonth,FirstDayOfNextYear,IsWeekday,IsHoliday,Seasonality,TipoDia,Wiso
0,20260101,2026-01-01,1,1st,Jueves,4,1,1,1,1,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01
1,20260102,2026-01-02,2,2nd,Viernes,5,1,1,1,2,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01
2,20260103,2026-01-03,3,3rd,Sábado,6,1,1,1,3,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,0,NaN,Sabado,2026-W01
3,20260104,2026-01-04,4,4th,Domingo,7,1,1,1,4,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,1,NaN,Domingo,2026-W01
4,20260105,2026-01-05,5,5th,Lunes,1,1,1,1,5,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W02


In [ ]:
# Limpieza tipo janitor::clean_names()
calendario.columns = calendario.columns.str.lower()

calendario.head()

,datekey,date,dayofmonth,daysuffix,dayname,dayofweek,dayofweekinmonth,dayofweekinyear,dayofquarter,dayofyear,...,lastdayofquarter,firstdayofyear,lastdayofyear,firstdayofnextmonth,firstdayofnextyear,isweekday,isholiday,seasonality,tipodia,wiso
0,20260101,2026-01-01,1,1st,Jueves,4,1,1,1,1,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01
1,20260102,2026-01-02,2,2nd,Viernes,5,1,1,1,2,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01
2,20260103,2026-01-03,3,3rd,Sábado,6,1,1,1,3,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,0,NaN,Sabado,2026-W01
3,20260104,2026-01-04,4,4th,Domingo,7,1,1,1,4,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,1,NaN,Domingo,2026-W01
4,20260105,2026-01-05,5,5th,Lunes,1,1,1,1,5,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W02


In [ ]:
# Conversión de tipos
calendario['date'] = pd.to_datetime(calendario['date'])
calendario['dayofweek'] = calendario['dayofweek'].astype(int)
calendario['weekofyear'] = calendario['weekofyear'].astype(int)

# Festivos (ya definidos antes)
calendario['isholiday'] = np.where(
    calendario['date'].isin(festivos),
    1,
    calendario['isholiday']
)

calendario.head()

,datekey,date,dayofmonth,daysuffix,dayname,dayofweek,dayofweekinmonth,dayofweekinyear,dayofquarter,dayofyear,...,lastdayofquarter,firstdayofyear,lastdayofyear,firstdayofnextmonth,firstdayofnextyear,isweekday,isholiday,seasonality,tipodia,wiso
0,20260101,2026-01-01,1,1st,Jueves,4,1,1,1,1,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,1,NaN,Habil,2026-W01
1,20260102,2026-01-02,2,2nd,Viernes,5,1,1,1,2,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01
2,20260103,2026-01-03,3,3rd,Sábado,6,1,1,1,3,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,0,NaN,Sabado,2026-W01
3,20260104,2026-01-04,4,4th,Domingo,7,1,1,1,4,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,1,NaN,Domingo,2026-W01
4,20260105,2026-01-05,5,5th,Lunes,1,1,1,1,5,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W02


In [ ]:
#Tipo día

calendario['tipodia'] = np.where(
    (calendario['dayofweek'].between(1, 5)) & (calendario['isholiday'] == 0),
    "Habil",
    "Sabado"
)

calendario['tipodia'] = np.where(
    calendario['isholiday'] == 1,
    "Festivo",
    calendario['tipodia']
)

# Nombre del día
map_dias = {
    1: "Lunes", 2: "Martes", 3: "Miercoles",
    4: "Jueves", 5: "Viernes", 6: "Sabado"
}

calendario['dayname'] = calendario['dayofweek'].map(map_dias).fillna("Domingo")

calendario.head()

,datekey,date,dayofmonth,daysuffix,dayname,dayofweek,dayofweekinmonth,dayofweekinyear,dayofquarter,dayofyear,...,lastdayofquarter,firstdayofyear,lastdayofyear,firstdayofnextmonth,firstdayofnextyear,isweekday,isholiday,seasonality,tipodia,wiso
0,20260101,2026-01-01,1,1st,Jueves,4,1,1,1,1,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,1,NaN,Festivo,2026-W01
1,20260102,2026-01-02,2,2nd,Viernes,5,1,1,1,2,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01
2,20260103,2026-01-03,3,3rd,Sabado,6,1,1,1,3,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,0,NaN,Sabado,2026-W01
3,20260104,2026-01-04,4,4th,Domingo,7,1,1,1,4,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,1,NaN,Festivo,2026-W01
4,20260105,2026-01-05,5,5th,Lunes,1,1,1,1,5,...,2026-03-31,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W02


In [ ]:
#Semana
calendario['semana'] = np.where(
    (calendario['date'].dt.month == 1) & (calendario['weekofyear'] >= 50),
    (calendario['date'].dt.year - 1).astype(str) + calendario['weekofyear'].astype(str),

    np.where(
        (calendario['date'].dt.month == 12) & (calendario['weekofyear'] <= 1),
        (calendario['date'].dt.year + 1).astype(str) + calendario['weekofyear'].astype(str),

        calendario['date'].dt.year.astype(str) + calendario['weekofyear'].astype(str)
    )
)

# Ajuste formato (equivalente nchar + str_sub)
calendario['semana'] = calendario['semana'].apply(
    lambda x: int(x[:4] + "0" + x[4]) if len(x) == 5 else int(x)
)

calendario.head()

,datekey,date,dayofmonth,daysuffix,dayname,dayofweek,dayofweekinmonth,dayofweekinyear,dayofquarter,dayofyear,...,firstdayofyear,lastdayofyear,firstdayofnextmonth,firstdayofnextyear,isweekday,isholiday,seasonality,tipodia,wiso,semana
0,20260101,2026-01-01,1,1st,Jueves,4,1,1,1,1,...,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,1,NaN,Festivo,2026-W01,202601
1,20260102,2026-01-02,2,2nd,Viernes,5,1,1,1,2,...,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01,202601
2,20260103,2026-01-03,3,3rd,Sabado,6,1,1,1,3,...,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,0,NaN,Sabado,2026-W01,202601
3,20260104,2026-01-04,4,4th,Domingo,7,1,1,1,4,...,2026-01-01,2026-12-31,2026-02-01,2027-01-01,False,1,NaN,Festivo,2026-W01,202601
4,20260105,2026-01-05,5,5th,Lunes,1,1,1,1,5,...,2026-01-01,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W02,202602


In [ ]:
#consecutivo de la semana
calendario['consec_tipo_dia_en_week'] = (
    calendario
    .groupby(['weekofyear', 'tipodia'])
    .cumcount() + 1
)

calendario.head()

,datekey,date,dayofmonth,daysuffix,dayname,dayofweek,dayofweekinmonth,dayofweekinyear,dayofquarter,dayofyear,...,lastdayofyear,firstdayofnextmonth,firstdayofnextyear,isweekday,isholiday,seasonality,tipodia,wiso,semana,consec_tipo_dia_en_week
0,20260101,2026-01-01,1,1st,Jueves,4,1,1,1,1,...,2026-12-31,2026-02-01,2027-01-01,True,1,NaN,Festivo,2026-W01,202601,1
1,20260102,2026-01-02,2,2nd,Viernes,5,1,1,1,2,...,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W01,202601,1
2,20260103,2026-01-03,3,3rd,Sabado,6,1,1,1,3,...,2026-12-31,2026-02-01,2027-01-01,False,0,NaN,Sabado,2026-W01,202601,1
3,20260104,2026-01-04,4,4th,Domingo,7,1,1,1,4,...,2026-12-31,2026-02-01,2027-01-01,False,1,NaN,Festivo,2026-W01,202601,2
4,20260105,2026-01-05,5,5th,Lunes,1,1,1,1,5,...,2026-12-31,2026-02-01,2027-01-01,True,0,NaN,Habil,2026-W02,202602,1


In [ ]:
#Secciones finales

calendario = calendario[['datekey', 'tipodia', 'semana', 'isholiday', 'dayofweek']]
calendario = calendario.rename(columns={'datekey': 'date_key'})

calendario['date_key'] = pd.to_datetime(calendario['date_key'])
calendario['mes'] = calendario['date_key'].dt.strftime('%Y%m').astype(int)

calendario.head()

,date_key,tipodia,semana,isholiday,dayofweek,mes
0,1970-01-01 00:00:00.020260101,Festivo,202601,1,4,197001
1,1970-01-01 00:00:00.020260102,Habil,202601,0,5,197001
2,1970-01-01 00:00:00.020260103,Sabado,202601,0,6,197001
3,1970-01-01 00:00:00.020260104,Festivo,202601,1,7,197001
4,1970-01-01 00:00:00.020260105,Habil,202602,0,1,197001


In [ ]:
hoy = pd.Timestamp.today().normalize()

# Asegurar tipo fecha correcto
calendario['date_key'] = pd.to_datetime(calendario['date_key']).dt.normalize()

filtro = calendario['date_key'] == hoy

if filtro.any():
    semana_actual = calendario.loc[filtro, 'semana'].iloc[0]
else:
    # fallback: última semana disponible
    semana_actual = calendario.loc[
        calendario['date_key'] <= hoy, 'semana'
    ].iloc[-1]

In [ ]:
listado_semanas = sorted(calendario['semana'].unique())

semana_estudio = calendario[
    calendario['semana'] == semana_actual
]
semana_estudio

,date_key,tipodia,semana,isholiday,dayofweek,mes
361,1970-01-01,Habil,202653,0,1,197001
362,1970-01-01,Habil,202653,0,2,197001
363,1970-01-01,Habil,202653,0,3,197001
364,1970-01-01,Habil,202653,0,4,197001


In [ ]:
#Parámetros

tipo_dia = "habil"
percentil = 0.70

dia_referente = (
    semana_estudio[
        semana_estudio['tipodia'] == tipo_dia.capitalize()
    ]['date_key'].iloc[0]
)

desde = semana_estudio['date_key'].min()
hasta = semana_estudio['date_key'].max()

vel_limite = 50
intervalo = 24.0
hora_inicio = 4
hora_fin = 23

week = semana_actual

In [ ]:
#archivos fms para revisión de información
vd_path = "Z:/01 base_datos/01 viajes_desglosados_FMS/"
ac_path = "Z:/01 base_datos/03 actividad_bus_FMS/"

# Listado de archivos
vd_list = [os.path.join(vd_path, f) for f in os.listdir(vd_path)]
ac_list = [os.path.join(ac_path, f) for f in os.listdir(ac_path)]

# DataFrames
vd_df = pd.DataFrame({'vd': vd_list})
vd_df['date_key'] = vd_df['vd'].str.extract(r'(\d{8})').astype(int)

ac_df = pd.DataFrame({'ac_bus': ac_list})
ac_df['date_key'] = ac_df['ac_bus'].str.extract(r'(\d{8})').astype(int)

# Merge solo con archivos necesarios
archivos = vd_df.merge(ac_df, on='date_key').dropna()

archivos

,vd,date_key,ac_bus
0,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20250421,Z:/01 base_datos/03 actividad_bus_FMS/20250421...
1,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20250422,Z:/01 base_datos/03 actividad_bus_FMS/20250422...
2,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20250423,Z:/01 base_datos/03 actividad_bus_FMS/20250423...
3,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20250424,Z:/01 base_datos/03 actividad_bus_FMS/20250424...
4,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20250425,Z:/01 base_datos/03 actividad_bus_FMS/20250425...
...,...,...,...
357,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20260418,Z:/01 base_datos/03 actividad_bus_FMS/20260418...
358,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20260419,Z:/01 base_datos/03 actividad_bus_FMS/20260419...
359,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20260420,Z:/01 base_datos/03 actividad_bus_FMS/20260420...
360,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20260421,Z:/01 base_datos/03 actividad_bus_FMS/20260421...


In [ ]:
# Definir rango en archivos
desde = 20260420
hasta = 20260422

In [ ]:
print("Fechas en archivos:", sorted(archivos['date_key'].unique()))
print("Desde:", desde)
print("Hasta:", hasta)

Fechas en archivos: [20250421, 20250422, 20250423, 20250424, 20250425, 20250426, 20250427, 20250428, 20250429, 20250430, 20250501, 20250502, 20250503, 20250504, 20250505, 20250506, 20250507, 20250508, 20250509, 20250510, 20250511, 20250512, 20250513, 20250514, 20250515, 20250516, 20250517, 20250518, 20250519, 20250520, 20250521, 20250522, 20250523, 20250524, 20250525, 20250526, 20250527, 20250529, 20250530, 20250531, 20250601, 20250602, 20250603, 20250604, 20250605, 20250606, 20250607, 20250608, 20250609, 20250610, 20250611, 20250612, 20250613, 20250614, 20250615, 20250616, 20250617, 20250618, 20250619, 20250620, 20250621, 20250622, 20250623, 20250624, 20250625, 20250626, 20250627, 20250628, 20250629, 20250630, 20250701, 20250702, 20250703, 20250704, 20250705, 20250706, 20250707, 20250708, 20250709, 20250710, 20250711, 20250712, 20250713, 20250714, 20250715, 20250716, 20250717, 20250718, 20250719, 20250720, 20250721, 20250722, 20250723, 20250724, 20250725, 20250726, 20250727, 20250728,

In [ ]:
archivos = archivos[
    (archivos['date_key'] >= desde) &
    (archivos['date_key'] <= hasta)
]

archivos

,vd,date_key,ac_bus
359,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20260420,Z:/01 base_datos/03 actividad_bus_FMS/20260420...
360,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20260421,Z:/01 base_datos/03 actividad_bus_FMS/20260421...
361,Z:/01 base_datos/01 viajes_desglosados_FMS/202...,20260422,Z:/01 base_datos/03 actividad_bus_FMS/20260422...


In [ ]:
ruta = "Z:/01 base_datos/55 informe velocidades/compilado/Velocidad_rutas_zonales.xlsx"

compilado_velocidades = pd.read_excel(
    ruta,
    sheet_name="vel_compilado"
)

# Equivalente a mutate + str_sub(-8, -1)
compilado_velocidades['Duracion'] = (
    compilado_velocidades['Duracion']
    .astype(str)
    .str[-8:]
)

compilado_velocidades.head()

,Str_linea,Str_ruta,Rutasae,Llave Ruta,Longitud,Velocidad,Duracion,Duracion_,Fecha_inicio,Fecha_fin,Semana
0,[7] SE10,[3704] SE10_20190311,3704,[7] SE10 - 3704,41,16,02:36:19,3,20191230,20200105,202001
1,[8] 128,[3791] 20190422_128_H,3791,[8] 128 - 3791,37,15,02:25:23,2,20191230,20200105,202001
2,[32] P500,[4080] 20191007_P500,4080,[32] P500 - 4080,33,17,01:53:22,2,20191230,20200105,202001
3,[74] 12,[3496] 12_20181126,3496,[74] 12 - 3496,41,19,02:07:25,2,20191230,20200105,202001
4,[76] 17-7,[157] 17-7,157,[76] 17-7 - 157,4,18,00:12:59,0,20191230,20200105,202001


In [ ]:
#procesamiento

ac_bus_compilado_aux = []

for i, row in archivos.iterrows():

    # VIAJES DESGLOSADOS
    vd = pd.read_csv(row['vd'], encoding='latin1')
    vd.columns = vd.columns.str.lower()

    vd = vd.rename(columns={
        'servicio': 'servicio_bus',
        'id línea ': 'linea'
    })

    vd['key_viaje'] = (
        vd['fecha'].astype(str) + "-" +
        vd['servicio_bus'].astype(str) + "-" +
        vd['id viaje '].astype(str)
    )

    vd = vd[['key_viaje', 'linea', 'ruta ']]


    # ACTIVIDAD BUS
    ac = pd.read_csv(row['ac_bus'], encoding='latin1')
    ac.columns = ac.columns.str.lower()

    # eliminar columnas tipo matches()
    drop_cols = [c for c in ac.columns if any(x in c for x in [
        'concesionario','codigo_bus','fms_bus','tabla',
        'viaje_linea','orden_viaje','tipo_de_nodo',
        'alarma_exceso_de_tiempo','conductor'
    ])]

    ac = ac.drop(columns=drop_cols, errors='ignore')

    ac = ac.rename(columns={'hora teórica': 'hora_teorica',
                            'hora llegada': 'hora_llegada',
                            'hora salida': 'hora_salida',
                            'servicio bus':'servicio_bus',
                            'id viaje': 'id_viaje',
                            'id línea':'id_l_nea',
                            'id nodo':'id_nodo',
                            'id ruta':'id_ruta'})

    # parseo de horas
    for col in ['hora_teorica', 'hora_llegada', 'hora_salida']:
        ac[col] = pd.to_datetime(ac[col], errors='coerce')

    ac['key_viaje'] = (
        ac['fecha'].astype(str) + "-" +
        ac['servicio_bus'].astype(str) + "-" +
        ac['id_viaje'].astype(str)
    )
    
    print(ac.head())

    # ordenar
    ac = ac.sort_values(['id_l_nea', 'servicio_bus', 'id_viaje', 'hora_teorica'])

    # consecutivo
    ac['consecutivo_parada'] = ac.groupby('key_viaje').cumcount() + 1
    ac['total_paradas'] = ac.groupby('key_viaje')['consecutivo_parada'].transform('max')

    # nodo_real
    ac['nodo_real'] = np.where(
        ac['evento'].str.contains("Fin Viaje", na=False),
        ac['id_nodo'].astype(str) + "-Fin Viaje",
        ac['id_nodo'].astype(str)
    )

    # h_real (minutos)
    def calc_hora(row):
        if pd.isna(row['hora_salida']) and pd.isna(row['hora_llegada']):
            return np.nan

        hora = row['hora_llegada'] if "Fin Viaje" in str(row['evento']) else row['hora_salida']

        if pd.isna(hora):
            return np.nan

        minutos = hora.hour * 60 + hora.minute + hora.second / 60

        if hora.day >= 2:
            minutos += 1440

        return minutos

    ac['h_real'] = ac.apply(calc_hora, axis=1)

    # lag / lead
    ac['h_real_lag'] = ac.groupby('key_viaje')['h_real'].shift(1)
    ac['h_real_lead'] = ac.groupby('key_viaje')['h_real'].shift(-1)

    # 🔹 limpieza tipo R
    ac['h_real'] = np.where(
        (ac['consecutivo_parada'] == 1) &
        (ac['h_real_lead'] > ac['h_real']),
        ac['h_real'],
        ac['h_real']
    )
    
    print(ac.head())

    # t_viaje
    ac['t_viaje'] = ac['h_real'] - ac['h_real_lag']

    ac.loc[(ac['h_real_lag'] > ac['h_real']), 't_viaje'] = np.nan
    ac.loc[(ac['t_viaje'] > 120), 't_viaje'] = np.nan

    # hora_paso
    ac['hora_paso'] = (ac['h_real'] // 60 // intervalo) * intervalo

    # join con vd
    ac = ac.merge(vd, on='key_viaje', how='left')

    # filtros
    ac = ac[(ac['total_paradas'] > 2) | (ac['evento'].isna())]

    ac['ruta_nodo'] = ac['id_ruta'].astype(str) + "-" + ac['nodo_real']

    ac = ac[
        (ac['h_real'] >= hora_inicio * 60) &
        (ac['h_real'] <= hora_fin * 60)
    ]

    ac = ac[['ruta_nodo', 'hora_paso', 't_viaje']]

    ac_bus_compilado_aux.append(ac)
    
    print(ac_bus_compilado_aux)
    

# 🔹 consolidado final
ac_bus_compilado_aux = pd.concat(ac_bus_compilado_aux, ignore_index=True)

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:26: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  ac = pd.read_csv(row['ac_bus'], encoding='latin1')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ac[col] = pd.to_datetime(ac[col], errors='coerce')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ac[col] = pd.to_datetime(ac[col], errors='coerce')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be 

        fecha    concesión  id_l_nea línea  id_ruta         ruta  viaje linea  \
0  20/04/2026  ENGATIVA ZN     10310  C101    12777  C101_Ida_V2            1   
1  20/04/2026  ENGATIVA ZN     10310  C101    12777  C101_Ida_V2            1   
2  20/04/2026  ENGATIVA ZN     10310  C101    12777  C101_Ida_V2            1   
3  20/04/2026  ENGATIVA ZN     10310  C101    12777  C101_Ida_V2            1   
4  20/04/2026  ENGATIVA ZN     10310  C101    12777  C101_Ida_V2            1   

   orden viaje  id_viaje  tipo de nodo  ...           evento  \
0            1         1             1  ...  Inicio Viaje(3)   
1            1         1             1  ...              NaN   
2            1         1             1  ...              NaN   
3            1         1             1  ...              NaN   
4            1         1             1  ...              NaN   

         hora_teorica hora referencia        hora_llegada         hora_salida  \
0 2026-04-23 05:48:30         5:48:30          

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:26: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  ac = pd.read_csv(row['ac_bus'], encoding='latin1')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ac[col] = pd.to_datetime(ac[col], errors='coerce')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ac[col] = pd.to_datetime(ac[col], errors='coerce')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be 

        fecha    concesión  id_l_nea  línea  id_ruta      ruta  viaje linea  \
0  21/04/2026  ENGATIVA ZN     10690  DL219    12721  DL219_V2            1   
1  21/04/2026  ENGATIVA ZN     10690  DL219    12721  DL219_V2            1   
2  21/04/2026  ENGATIVA ZN     10690  DL219    12721  DL219_V2            1   
3  21/04/2026  ENGATIVA ZN     10690  DL219    12721  DL219_V2            1   
4  21/04/2026  ENGATIVA ZN     10690  DL219    12721  DL219_V2            1   

   orden viaje  id_viaje  tipo de nodo  ...           evento  \
0            1         2             1  ...  Inicio Viaje(3)   
1            1         2             1  ...              NaN   
2            1         2             1  ...              NaN   
3            1         2             1  ...              NaN   
4            1         2             1  ...              NaN   

         hora_teorica hora referencia hora_llegada  hora_salida  \
0 2026-04-23 15:24:30        15:24:30          NaT          NaT   
1 2026

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:26: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  ac = pd.read_csv(row['ac_bus'], encoding='latin1')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ac[col] = pd.to_datetime(ac[col], errors='coerce')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ac[col] = pd.to_datetime(ac[col], errors='coerce')
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_10636\1175015872.py:49: UserWarning: Could not infer format, so each element will be 

        fecha    concesión  id_l_nea línea  id_ruta        ruta  viaje linea  \
0  22/04/2026  ENGATIVA ZN     10264   614    12327  614_Ida_V2            1   
1  22/04/2026  ENGATIVA ZN     10264   614    12327  614_Ida_V2            1   
2  22/04/2026  ENGATIVA ZN     10264   614    12327  614_Ida_V2            1   
3  22/04/2026  ENGATIVA ZN     10264   614    12327  614_Ida_V2            1   
4  22/04/2026  ENGATIVA ZN     10264   614    12327  614_Ida_V2            1   

   orden viaje  id_viaje  tipo de nodo  ...           evento  \
0            1         5             1  ...  Inicio Viaje(3)   
1            1         5             1  ...              NaN   
2            1         5             1  ...              NaN   
3            1         5             1  ...              NaN   
4            1         5             1  ...              NaN   

         hora_teorica hora referencia hora_llegada  hora_salida  \
0 2026-04-23 19:56:15        19:56:15          NaT          NaT   


In [ ]:
ac

,ruta_nodo,hora_paso,t_viaje


In [ ]:
vd

,key_viaje,linea,ruta
0,22/04/2026-AD0264052-5,10264,12327
1,22/04/2026-AD0264052-3,10264,12328
2,22/04/2026-AD0550014-1,10550,11081
3,22/04/2026-BC29D0002-2,10350,12739
4,22/04/2026-BC29D0002-3,10350,12738
...,...,...,...
3308,22/04/2026-CN232G017-1,10339,12756
3309,22/04/2026-CN232G017-2,10339,12756
3310,22/04/2026-CN232G017-3,10339,12756
3311,22/04/2026-CN232G018-1,10339,12756


In [ ]:
ac_bus_compilado_aux

,ruta_nodo,hora_paso,t_viaje
